# 01 · Bronze: historical job ads

Backfill of historical job ads from the **JobTech Historical Ads API** into the lakehouse `lh_bronze`.

- **Input:** `https://historical.api.jobtechdev.se/search`, one request window per search term and month.
- **Output:** the raw API responses, unchanged, stored as
  `Files/bronze/historical/term=<term>/year=<YYYY>/month=<MM>/ads.json`,
  each with a `_meta` block (request parameters, fetch time, total reported by the API, number of ads saved).
- **How to run:** *Run all*. The notebook is safe to re-run: months that are already stored are skipped.

## Design notes

- **Monthly request windows.** The API does not paginate beyond offset 2000, so each request covers one term and one month (the largest month so far has well under 2000 ads).
- **Hive-style partition folders** (`term=…/year=…/month=…`) let Spark read term, year and month as columns in the silver layer.
- **Idempotent.** Existing months are skipped. Existence is checked against OneLake with `notebookutils`, because the local `/lakehouse/default` mount can report stale results after files have been deleted.
- **Search terms.** Terms the API recognises as occupations (e.g. *data engineer*) are matched precisely. Unrecognised multi-word terms fall back to matching *any* of the words, so they must be quoted as phrases: `"analytics engineer"` gave 80 ads in 2025, unquoted it gave 21,162.
- **Overlap between terms is expected.** The same ad can be found via several terms; ads are deduplicated on id in the silver layer.
- **Verification.** The last cell fails loudly if any month is missing or if the number of saved ads differs from what the API reported.


## 1. Parameters

In [1]:
# Which search terms and months to fetch.
# END is the last complete month; the current month is handled by the daily pipeline.
SEARCH_TERMS = [
    "data engineer", "data scientist", "dataanalytiker", "data analyst",
    "BI-utvecklare", "BI developer", '"analytics engineer"',
]
START = (2016, 1)   # (year, month)
END = (2026, 8)     # (year, month), inclusive

StatementMeta(, 28fa3142-ab98-4d96-96b7-ea68399e4888, 3, Finished, Available, Finished, False)

## 2. Setup and helper functions

In [2]:
import json
import time
from calendar import monthrange
from datetime import datetime, timezone
from pathlib import Path

import requests

HIST_URL = "https://historical.api.jobtechdev.se/search"
PAGE_SIZE = 100
MAX_OFFSET = 2000       # the API rejects offsets above this
PAUSE_SECONDS = 0.3     # be polite to the API between pages

BRONZE_REL = "Files/bronze/historical"        # path as OneLake sees it
LAKEHOUSE_ROOT = Path("/lakehouse/default")   # local mount of the default lakehouse


def months(start, end):
    """Yield (year, month) from start to end, inclusive."""
    y, m = start
    while (y, m) <= end:
        yield y, m
        y, m = (y + 1, 1) if m == 12 else (y, m + 1)


def slug(term):
    """'"analytics engineer"' -> 'analytics_engineer' (safe folder name)."""
    return term.lower().replace('"', "").replace(" ", "_").replace("-", "_")


def rel_path(term, year, month):
    """OneLake path of the file for one search term and month."""
    return f"{BRONZE_REL}/term={slug(term)}/year={year}/month={month:02d}/ads.json"


def fetch_page(params, retries=3):
    """One API call, retried with a short back-off on temporary errors."""
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(HIST_URL, params=params, timeout=60)
            r.raise_for_status()
            return r.json()
        except requests.RequestException as e:
            print(f"  Attempt {attempt} failed: {e}")
            if attempt == retries:
                raise
            time.sleep(2 * attempt)


def fetch_month(term, year, month):
    """Fetch all ads for one search term and month, page by page."""
    last_day = monthrange(year, month)[1]
    base = {
        "q": term,
        "published-after": f"{year}-{month:02d}-01T00:00:00",
        "published-before": f"{year}-{month:02d}-{last_day}T23:59:59",
        "limit": PAGE_SIZE,
        # Stricter matching of free-text words. Does NOT fix unrecognised phrases:
        # quote those in SEARCH_TERMS instead.
        "x-feature-freetext-bool-method": "and",
    }
    hits, offset, total = [], 0, None
    while True:
        data = fetch_page({**base, "offset": offset})
        total = data["total"]["value"]
        page = data.get("hits", [])
        hits.extend(page)
        offset += PAGE_SIZE
        if not page or offset >= total:
            break
        if offset > MAX_OFFSET:
            print(f"  WARNING: {total} hits but the API only pages to ~{MAX_OFFSET}; saved {len(hits)}.")
            break
        time.sleep(PAUSE_SECONDS)
    return base, total, hits

StatementMeta(, 28fa3142-ab98-4d96-96b7-ea68399e4888, 4, Finished, Available, Finished, False)

## 3. Fetch and store

In [3]:
for term in SEARCH_TERMS:
    fetched = skipped = ads = 0
    for year, month in months(START, END):
        rp = rel_path(term, year, month)

        # Idempotency: skip months that are already stored.
        # Checked against OneLake directly; the local mount can be stale.
        if notebookutils.fs.exists(rp):
            skipped += 1
            continue

        params, total, hits = fetch_month(term, year, month)
        record = {
            "_meta": {
                "source": "jobtech_historical_api",
                "search_term": term,
                "params": params,
                "fetched_at_utc": datetime.now(timezone.utc).isoformat(),
                "total_reported": total,
                "n_hits": len(hits),
            },
            "hits": hits,
        }
        out_file = LAKEHOUSE_ROOT / rp
        out_file.parent.mkdir(parents=True, exist_ok=True)
        out_file.write_text(json.dumps(record, ensure_ascii=False), encoding="utf-8")
        fetched += 1
        ads += len(hits)

    print(f"{term:22} fetched {fetched:3} months ({ads:5} ads), skipped {skipped:3} already stored")

StatementMeta(, 28fa3142-ab98-4d96-96b7-ea68399e4888, 5, Finished, Available, Finished, False)

data engineer          fetched   0 months (    0 ads), skipped 128 already stored
data scientist         fetched   0 months (    0 ads), skipped 128 already stored
dataanalytiker         fetched   0 months (    0 ads), skipped 128 already stored
data analyst           fetched   0 months (    0 ads), skipped 128 already stored
BI-utvecklare          fetched   0 months (    0 ads), skipped 128 already stored
BI developer           fetched   0 months (    0 ads), skipped 128 already stored
"analytics engineer"   fetched   0 months (    0 ads), skipped 128 already stored


## 4. Verify

In [4]:
expected = list(months(START, END))
missing, mismatched = [], []
n_files = n_ads = 0

for term in SEARCH_TERMS:
    for year, month in expected:
        rp = rel_path(term, year, month)
        if not notebookutils.fs.exists(rp):
            missing.append((term, f"{year}-{month:02d}"))
            continue
        meta = json.loads((LAKEHOUSE_ROOT / rp).read_text(encoding="utf-8"))["_meta"]
        n_files += 1
        n_ads += meta["n_hits"]
        if meta["n_hits"] != meta["total_reported"]:
            mismatched.append((term, f"{year}-{month:02d}", meta["n_hits"], meta["total_reported"]))

print(f"Files: {n_files} of {len(SEARCH_TERMS) * len(expected)} expected")
print(f"Ads (incl. overlap between terms): {n_ads}")
print(f"Missing months: {len(missing)}", missing[:10])
print(f"Count mismatches: {len(mismatched)}", mismatched[:10])

# Fail loudly so that a scheduled run shows up as failed if something is wrong.
assert not missing and not mismatched, "Bronze verification failed"

StatementMeta(, 28fa3142-ab98-4d96-96b7-ea68399e4888, 6, Finished, Available, Finished, False)

Files: 896 of 896 expected
Ads (incl. overlap between terms): 16458
Missing months: 0 []
Count mismatches: 0 []
